# JA-NET — mixed-solvent solubility from molecular structure

A neural network with the **Jouyban-Acree** cosolvency equation built in: it predicts a solute's solubility
across a binary-solvent composition curve from SMILES alone. This notebook runs the core evaluation and draws
a predicted cosolvency curve.

Requires `data/mixturesoldb.csv` (see `data/README.md`); the first run builds a feature cache (a few minutes).

In [ ]:
from janet import load_features, evaluate, demo_cosolvency_curve
import numpy as np, matplotlib.pyplot as plt

D = load_features()
print(f'{len(D.y)} rows | {len(set(D.sysid))} binary systems | {len(set(D.soluteid))} solutes')

## Headline: generalization to unseen solvent systems

Leave-binary-system-out (each solute-solvent system held out entirely). Expected: **JA-NET MAE ~ 0.32**,
about 15% better than the concatenation baselines (repeated-split significance is in `experiments/rigor_eval.py`).

In [ ]:
evaluate(D, 'LEAVE-BINARY-SYSTEM-OUT', D.sysid);

## Honest boundary: novel solute chemistry

Leave-solute-out (brand-new solutes). Here gradient boosting is competitive - the Jouyban-Acree structure
encodes *composition* generalization, not solute novelty. Reporting the boundary is the point.

In [ ]:
evaluate(D, 'LEAVE-SOLUTE-OUT', D.soluteid);

## A cosolvency curve predicted from structure

Hold one system out entirely and predict its full solubility-vs-composition curve from SMILES alone;
the black points were never seen in training. The log-linear line is what you get with no cosolvency term.

In [ ]:
r = demo_cosolvency_curve(D)

plt.figure(figsize=(6, 4))
plt.scatter(r['mx'], r['my'], c='k', zorder=5, label='measured (held out)')
plt.plot(r['xs'], r['pred'], lw=2, label='JA-NET (from structure)')
plt.plot(r['xs'], r['loglin'], '--', label='log-linear (no cosolvency)')
plt.xlabel('solvent-1 fraction'); plt.ylabel('logS (mole fraction)')
plt.title(r['name'][:40]); plt.legend(); plt.tight_layout(); plt.show()